In [1]:
import os
from datasets import load_dataset
from dotenv import load_dotenv
import chess
import math
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import numpy as np
from itertools import islice
import zstandard as zstd

load_dotenv()

/home/nkminion/miniconda3/envs/PyTorchVenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
TOKEN = os.getenv("HFREAD")

if TOKEN is None:
	raise ValueError("Token not found")

Dataset = load_dataset(
	"Lichess/chess-position-evaluations",
	split='train',
	streaming=True,
	token=TOKEN
)

print('Dataset Loaded')

Dataset Loaded


In [3]:
def ProcessChessData(FENString,CPScore,MateScore):
	tensor = np.zeros((16,8,8), dtype=np.float32)
	board = chess.Board(FENString)

	PieceToLayer = {
		'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
		'p':6,'n':7,'b':8,'r':9,'q':10,'k':11
	}

	for square in chess.SQUARES:
		piece = board.piece_at(square)

		if piece:
			symbol = piece.symbol()
			layer = PieceToLayer[symbol]

			row = 7-(square//8)
			col = square%8

			tensor[layer,row,col] = 1.0

	if board.turn == chess.WHITE:
		tensor[12,:,:] = 1.0

	if board.has_kingside_castling_rights(chess.WHITE):
		tensor[13,7,7] = 1.0
	if board.has_queenside_castling_rights(chess.WHITE):
		tensor[13,7,0] = 1.0
	if board.has_kingside_castling_rights(chess.BLACK):
		tensor[13,0,7] = 1.0
	if board.has_queenside_castling_rights(chess.BLACK):
		tensor[13,0,0] = 1.0

	for i in range(8):
		tensor[14,i,:] = (1.0/7)*(i)
		tensor[15,:,i] = (1.0/7)*(i)

	if MateScore is not None:
		TargetScore = 1.0 if MateScore > 0 else -1.0
	else:
		CPScore = CPScore if CPScore is not None else 0
		TargetScore = math.tanh(CPScore/200.0)

	TargetScore = np.array([TargetScore],dtype=np.float32)

	return tensor,TargetScore

In [4]:
class ChessIterableDataset(IterableDataset):
	def __init__(self,hfDataset):
		self.hfDataset = hfDataset

	def __iter__(self):
		WorkerInfo = get_worker_info()
		if WorkerInfo is None:
			start = 0
			step = 1
		else:
			start = WorkerInfo.id
			step = WorkerInfo.num_workers

		ShardedStream = islice(self.hfDataset,start,None,step)

		for data in ShardedStream:
			inputs,target = ProcessChessData(data['fen'],data['cp'],data['mate'])

			yield inputs,target

In [5]:
TrainDataset = ChessIterableDataset(Dataset)
TrainLoader = DataLoader(
	TrainDataset,
	batch_size=1024,
	num_workers=4,
	pin_memory=True,
	prefetch_factor=1,
	persistent_workers=True
)
Total = 0
with open('Boards.bin','wb') as BoardFile:
	with open('Targets.bin','wb') as TargetFile:
		for BatchIdx,(inputs,targets) in enumerate(TrainLoader):
			inputs = inputs.numpy().tobytes()
			targets = targets.numpy().tobytes()
			BoardFile.write(inputs)
			TargetFile.write(targets)
			Total += len(targets)
			if (BatchIdx+1) % 1000 == 0:
				print(f'Processed {BatchIdx+1} Batches')

print(f'Completed! | Total Batches: {Total//1024} | Number of boards: {Total}')

Processed 1000 Batches
Processed 2000 Batches
Processed 3000 Batches
Processed 4000 Batches
Processed 5000 Batches
Processed 6000 Batches
Processed 7000 Batches
Processed 8000 Batches
Processed 9000 Batches
Processed 10000 Batches
Processed 11000 Batches
Processed 12000 Batches
Processed 13000 Batches
Processed 14000 Batches
Processed 15000 Batches
Processed 16000 Batches
Processed 17000 Batches
Processed 18000 Batches
Processed 19000 Batches
Processed 20000 Batches
Processed 21000 Batches
Processed 22000 Batches
Processed 23000 Batches
Processed 24000 Batches
Processed 25000 Batches
Processed 26000 Batches
Processed 27000 Batches
Processed 28000 Batches
Processed 29000 Batches
Processed 30000 Batches


KeyboardInterrupt: 

Downloaded around 120GiB of tensors. Stopped it midway.

In [2]:
cctx = zstd.ZstdCompressor(level=10)

with open('Boards.bin','rb') as BoardFile:
	with open('Targets.bin','rb') as TargetFile:
		with open ('ChessData.zst','wb') as ResultFile:
			with cctx.stream_writer(ResultFile) as stream:
				while True:
					BoardBytes = BoardFile.read(4096)
					TargetBytes = TargetFile.read(4)

					if not BoardBytes or not TargetBytes:
						break

					stream.write(BoardBytes+TargetBytes)

print('Compression Complete!')

Compression Complete!


Final size = 1.2GiB